In [ ]:
from collections import Counter
import clslib
import utils
from datasets import load_dataset
import pandas as pd
import zipfile
import pickle
import urllib.request
from nltk.corpus import opinion_lexicon
import nltk
from itertools import combinations, product


In [ ]:
# load dataset
dataset = load_dataset("stanfordnlp/sst2")

train = dataset["train"].to_pandas()
heldout = dataset["validation"].to_pandas()

In [17]:
# load fasttext embedding model
fasttext = utils.load_pretrained_embeddings("fasttext")

Loaded pretrained embeddings: fasttext



In [ ]:
# load tokenizer and coverage analysis function
tokenizer = clslib.BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False,
    vocab=fasttext.vocab
)

def vocabulary_analysis(df, emb):
    counts = Counter(
        token
        for sentence in df["sentence"]
        for token in tokenizer.tokens(sentence)
    )

    oov = Counter({
        word: freq
        for word, freq in counts.items()
        if word not in emb
    })

    print(f"Vocabulary size: {len(counts)}")
    print(f"Type coverage: {1 - len(oov) / len(counts):.2%}")
    print(f"Token coverage: {1 - sum(oov.values()) / sum(counts.values()):.2%}")
    print(f"Top 15 unknown tokens: {oov.most_common(15)}")

In [ ]:
# check coverage for fasttext
print("TRAIN")
vocabulary_analysis(train, fasttext)

print("\nHELD-OUT")
vocabulary_analysis(heldout, fasttext)

TRAIN
Vocabulary size: 14801
Type coverage: 95.68%
Token coverage: 98.62%
Top 15 unknown tokens: [("n't", 2311), ('``', 1757), ('`', 970), ("''", 43), ('and/or', 27), ('1/2', 25), ('writer/director', 23), ('clockstoppers', 22), ('not-very-funny', 19), ('sequel-for-the-sake', 16), ('still-inestimable', 15), ('nebrida', 15), ('fun-for-fun', 15), ('clung-to', 15), ('opera-ish', 15)]

HELD-OUT
Vocabulary size: 4337
Type coverage: 97.81%
Token coverage: 98.72%
Top 15 unknown tokens: [("n't", 72), ('``', 32), ('`', 20), ('seldahl', 2), ('clockstoppers', 2), ('lifetime-channel', 1), ('birot', 1), ('slash-and-hack', 1), ('alientation', 1), ('pussy-ass', 1), ('killer-thrillers', 1), ('old-form', 1), ('montias', 1), ('foot-age', 1), ('less-than-objective', 1)]


In [ ]:
# fasttexg doesn't have don't or n't. The negation don't is not embedded properly
for word in ["not", "don't", "n't"]:
    print(word, word in fasttext)

not True
don't False
n't False


In [ ]:
# load glove embedding model

url = "https://surfdrive.surf.nl/s/qYfZ6PiNrMLCwpS/download"
filename = "glove.2024.wikigiga.50d.zip"

urllib.request.urlretrieve(url, filename)

path, dims, skip_first = utils.download_pretrained("glove")

glove = clslib.PretrainedEmbeddings(
    path, dims, skip_first_line=skip_first
)

print("Vocabulary size:", len(glove.vocab))
print("Embedding dimensions:", glove.num_dims)

Vocabulary size: 1287615
Embedding dimensions: 50


In [11]:
# check coverage for glove
print("TRAIN")
vocabulary_analysis(train, glove)

print("\nHELD-OUT")
vocabulary_analysis(heldout, glove)

TRAIN
Vocabulary size: 14801
Type coverage: 90.64%
Token coverage: 98.30%
Top 15 unknown tokens: [('old-fashioned', 121), ('sci-fi', 96), ('coming-of-age', 94), ('writer-director', 75), ('first-rate', 71), ('well-made', 71), ('rip-off', 61), ('thought-provoking', 61), ('gross-out', 57), ('feel-good', 55), ('low-budget', 52), ('self-conscious', 51), ('well-crafted', 47), ('self-indulgent', 44), ('heavy-handed', 41)]

HELD-OUT
Vocabulary size: 4337
Type coverage: 95.11%
Token coverage: 98.64%
Top 15 unknown tokens: [('heavy-handed', 5), ('well-acted', 4), ('writer-director', 3), ('thought-provoking', 3), ('self-conscious', 2), ('after-school', 2), ('oscar-winning', 2), ('by-the-numbers', 2), ('live-action', 2), ('first-time', 2), ('hip-hop', 2), ('sci-fi', 2), ('follow-up', 2), ('high-spirited', 1), ('morning-glory', 1)]


In [12]:
for word in ["not", "don't", "n't"]:
    print(word, word in glove)

not True
don't False
n't True


In [ ]:
words = [
    "old-fashioned", "sci-fi", "coming-of-age",
    "writer-director", "first-rate", "well-made",
    "thought-provoking", "feel-good",
    "self-conscious", "heavy-handed"
]

for word in words:
    print(f"{word:25} FastText: {word in fasttext} | GloVe: {word in glove}")

old-fashioned             FastText: True | GloVe: False
sci-fi                    FastText: True | GloVe: False
coming-of-age             FastText: True | GloVe: False
writer-director           FastText: True | GloVe: False
first-rate                FastText: True | GloVe: False
well-made                 FastText: True | GloVe: False
thought-provoking         FastText: True | GloVe: False
feel-good                 FastText: True | GloVe: False
self-conscious            FastText: True | GloVe: False
heavy-handed              FastText: True | GloVe: False


In [ ]:
nltk.download("opinion_lexicon", quiet=True)

# Count word frequencies in SST-2 training data
counts = Counter(
    word
    for sentence in train["sentence"]
    for word in tokenizer.tokens(sentence)
)

# Positive and negative word lists
positive = set(opinion_lexicon.positive())
negative = set(opinion_lexicon.negative())

# Exclude words appearing in both lists
positive -= positive & negative
negative -= set(opinion_lexicon.positive())

# Keep words covered by both embedding models
common_words = {
    word for word in counts
    if word in fasttext and word in glove
}

positive_top = [
    (word, freq)
    for word, freq in counts.most_common()
    if word in positive and word in common_words
][:10]

negative_top = [
    (word, freq)
    for word, freq in counts.most_common()
    if word in negative and word in common_words
][:10]

print("POSITIVE:")
print(positive_top)

print("\nNEGATIVE:")
print(negative_top)

POSITIVE:
[('like', 1782), ('good', 1341), ('best', 807), ('love', 777), ('work', 761), ('enough', 743), ('humor', 660), ('well', 624), ('great', 617), ('better', 532)]

NEGATIVE:
[('funny', 1105), ('bad', 907), ('plot', 563), ('hard', 368), ('dull', 285), ('worst', 271), ('dark', 239), ('silly', 210), ('boring', 173), ('dumb', 167)]


In [ ]:
# filtering of the top positive and negative words
positive = ["good", "best", "love", "great", "better"]
negative = ["bad", "worst", "dull", "boring", "dumb"]

def cosine(w1, w2, model):
    u = model.embed_word(w1)
    v = model.embed_word(w2)
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

def analyse_similarity(model):
    pairs = {
        "Positive–Positive": list(combinations(positive, 2)),
        "Negative–Negative": list(combinations(negative, 2)),
        "Positive–Negative": list(product(positive, negative))
    }

    for category, word_pairs in pairs.items():
        scores = [cosine(a, b, model) for a, b in word_pairs]
        print(f"{category}: {np.mean(scores):.4f}")

print("FASTTEXT")
analyse_similarity(fasttext)

print("\nGLOVE")
analyse_similarity(glove)

FASTTEXT
Positive–Positive: 0.5590
Negative–Negative: 0.5485
Positive–Negative: 0.4837

GLOVE
Positive–Positive: 0.8062
Negative–Negative: 0.5843
Positive–Negative: 0.5515


In [19]:
pos_lex = set(opinion_lexicon.positive())
neg_lex = set(opinion_lexicon.negative())

# Words appearing in SST-2 and covered by both models
shared = {w for w in counts if w in fasttext and w in glove}

positive = sorted((pos_lex - neg_lex) & shared)
negative = sorted((neg_lex - pos_lex) & shared)

print("Positive words:", len(positive))
print("Negative words:", len(negative))

Positive words: 961
Negative words: 1602


In [21]:
print("FASTTEXT")
analyse_similarity(fasttext)

print("\nGLOVE")
analyse_similarity(glove)

FASTTEXT
Positive–Positive: 0.3321
Negative–Negative: 0.3210
Positive–Negative: 0.2966

GLOVE
Positive–Positive: 0.3229
Negative–Negative: 0.2682
Positive–Negative: 0.2390


In [20]:
# SST-2 training vocabulary
vocab = set(
    word
    for sentence in train["sentence"]
    for word in tokenizer.tokens(sentence)
)

# Common vocabulary
common = {w for w in vocab if w in fasttext and w in glove}

# Sentiment lexicon (exclude overlapping labels)
pos_lex = set(opinion_lexicon.positive())
neg_lex = set(opinion_lexicon.negative())

pos_words = sorted((pos_lex - neg_lex) & common)
neg_words = sorted((neg_lex - pos_lex) & common)

print("Positive words:", len(pos_words))
print("Negative words:", len(neg_words))


def normalized_vectors(words, model):
    X = np.stack([model.embed_word(w) for w in words])
    return X / np.linalg.norm(X, axis=1, keepdims=True)


def sentiment_similarity(model):
    P = normalized_vectors(pos_words, model)
    N = normalized_vectors(neg_words, model)

    # Average similarity of distinct word pairs
    pp = ((P @ P.T).sum() - len(P)) / (len(P) * (len(P) - 1))
    nn = ((N @ N.T).sum() - len(N)) / (len(N) * (len(N) - 1))
    pn = (P @ N.T).mean()

    print(f"Positive–Positive: {pp:.4f}")
    print(f"Negative–Negative: {nn:.4f}")
    print(f"Positive–Negative: {pn:.4f}")
    print(f"Separation:        {(pp + nn)/2 - pn:.4f}")


print("\nFASTTEXT")
sentiment_similarity(fasttext)

print("\nGLOVE")
sentiment_similarity(glove)

Positive words: 961
Negative words: 1602

FASTTEXT
Positive–Positive: 0.3321
Negative–Negative: 0.3210
Positive–Negative: 0.2966
Separation:        0.0299

GLOVE
Positive–Positive: 0.3229
Negative–Negative: 0.2682
Positive–Negative: 0.2390
Separation:        0.0565


In [ ]:
# load the imdb embeddings
with zipfile.ZipFile("data.zip") as z:
    def imdb_file(filename):
        return next(
            p for p in z.namelist()
            if "/imdb/" in "/" + p.lower()
            and p.lower().endswith(filename)
        )

    words = [
        line.split()[0]
        for line in z.read(imdb_file("wordlist.txt")).decode("utf-8").splitlines()
        if line.strip()
    ]

    vectors = pickle.loads(
        z.read(imdb_file("embinit.save")),
        encoding="latin1"
    )

vectors = np.asarray(vectors, dtype=np.float32)

print("Vocabulary size:", len(words))
print("Matrix shape:", vectors.shape)

Vocabulary size: 105373
Matrix shape: (105374, 200)


In [24]:
# convert and load using clslib
with open("imdb_vectors.txt", "w", encoding="utf-8") as f:
    for word, vector in zip(words, vectors):
        f.write(word + " " + " ".join(map(str, vector)) + "\n")

imdb = clslib.PretrainedEmbeddings(
    "imdb_vectors.txt",
    num_dims=vectors.shape[1]
)

In [27]:
# check coverage for imdb
print("TRAIN")
vocabulary_analysis(train, imdb)

print("\nHELD-OUT")
vocabulary_analysis(heldout, imdb)

TRAIN
Vocabulary size: 14801
Type coverage: 93.40%
Token coverage: 98.81%
Top 15 unknown tokens: [(')', 1179), ('(', 1074), ('not-very-funny', 19), ('nijinsky', 19), ('sequel-for-the-sake', 16), ('self-caricature', 16), ('backmasking', 16), ('payami', 16), ('still-inestimable', 15), ('nebrida', 15), ('fun-for-fun', 15), ('rubbo', 15), ('clung-to', 15), ('kosashvili', 14), ('pop-induced', 14)]

HELD-OUT
Vocabulary size: 4337
Type coverage: 96.50%
Token coverage: 98.59%
Top 15 unknown tokens: [('(', 43), (')', 43), ('elling', 2), ('chaiken', 2), ('seldahl', 2), ('serry', 2), ('morning-glory', 1), ('iditarod', 1), ('125-year', 1), ('hile', 1), ('lifetime-channel', 1), ('anne-sophie', 1), ('birot', 1), ('foot-dragging', 1), ('slash-and-hack', 1)]


In [32]:
# Keep only words with nonzero IMDb embeddings
valid_words = {
    w for w in positive + negative
    if np.linalg.norm(imdb.embed_word(w)) > 0
}

positive = [w for w in positive if w in valid_words]
negative = [w for w in negative if w in valid_words]

print("Positive:", len(positive))
print("Negative:", len(negative))

for name, model in [
    ("FastText", fasttext),
    ("GloVe", glove),
    ("IMDb", imdb)
]:
    print(f"\n{name}")
    analyse_similarity(model)

Positive: 660
Negative: 913

FastText
Positive–Positive: 0.3528
Negative–Negative: 0.3396
Positive–Negative: 0.3139

GloVe
Positive–Positive: 0.3962
Negative–Negative: 0.3410
Positive–Negative: 0.3055

IMDb
Positive–Positive: 0.2308
Negative–Negative: 0.1919
Positive–Negative: 0.1717


In [29]:
for word in ["not", "don't", "n't"]:
    print(word, word in imdb)

not True
don't False
n't True


In [33]:
for name, df in [("Train", train), ("Held-out", heldout)]:
    counts = Counter(
        word
        for sentence in df["sentence"]
        for word in tokenizer.tokens(sentence)
    )

    oov = {w for w in counts if w not in imdb}
    zero = {
        w for w in counts
        if w in imdb and not np.any(imdb.embed_word(w))
    }

    total_tokens = sum(counts.values())
    unusable_tokens = sum(counts[w] for w in oov | zero)

    print(f"\n{name}")
    print(f"Original type coverage: {1 - len(oov)/len(counts):.2%}")
    print(f"Effective type coverage: {1 - len(oov | zero)/len(counts):.2%}")
    print(f"Effective token coverage: {1 - unusable_tokens/total_tokens:.2%}")
    print(f"OOV types: {len(oov)}")
    print(f"Zero-vector types: {len(zero)}")
    print(f"Zero-vector occurrences: {sum(counts[w] for w in zero)}")
    print("Most frequent zero-vector words:",
          sorted(zero, key=lambda w: counts[w], reverse=True)[:15])


Train
Original type coverage: 93.40%
Effective type coverage: 48.28%
Effective token coverage: 90.58%
OOV types: 977
Zero-vector types: 6678
Zero-vector occurrences: 52158
Most frequent zero-vector words: ['meditation', 'listless', 'retread', 'rhythms', 'buoyant', 'eloquent', 'unfocused', 'joyous', 'intoxicating', 'wickedly', 'deft', 'charmer', 'kiddie', 'revelatory', 'endeavor']

Held-out
Original type coverage: 96.50%
Effective type coverage: 69.17%
Effective token coverage: 91.11%
OOV types: 152
Zero-vector types: 1185
Zero-vector occurrences: 1276
Most frequent zero-vector words: ['exquisitely', 'pinocchio', 'confirms', 'spontaneity', 'sade', 'downer', 'dizzying', 'pumpkin', 'volatile', 'beguiling', 'splash', 'frailty', 'carnahan', 'seeds', 'carvey']


In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

words = positive + negative
y = np.array([1] * len(positive) + [0] * len(negative))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for name, model in [("FastText", fasttext), ("GloVe", glove)]:
    X = np.stack([model.embed_word(w) for w in words])

    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    scores = cross_val_score(
        classifier, X, y, cv=cv, scoring="balanced_accuracy"
    )

    print(f"{name}: {scores.mean():.4f} ± {scores.std():.4f}")

FastText: 0.9399 ± 0.0106
GloVe: 0.9073 ± 0.0194
